# Calibración de threshold orientada a seguridad

Este notebook revisa los artefactos generados por `src.evaluation.calibrate_security_thresholds`. Los thresholds se seleccionaron exclusivamente con validation; test limpio y stress se muestran solo como evaluación posterior. No entrena ni carga el modelo.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
result_dir = project_root / 'outputs' / 'experiments' / 'baseline_formal' / 'baseline_con_aumento' / 'security_threshold_calibration'
comparison = pd.read_csv(result_dir / 'security_threshold_comparison.csv')
stress = pd.read_csv(result_dir / 'stress_metrics_by_candidate.csv')
with (result_dir / 'security_threshold_calibration.json').open(encoding='utf-8') as file:
    summary = json.load(file)
summary['recommendations']

## Comparación de candidatos

La tabla permite revisar el intercambio entre FAR (riesgo de aceptar impostores) y FRR (fricción para usuarios legítimos).

In [ ]:
columns = [
    'criterion', 'threshold', 'validation_far', 'validation_frr',
    'validation_f1', 'test_far', 'test_frr', 'test_f1',
    'test_accuracy', 'test_fp', 'test_fn', 'worst_stress'
]
table = comparison[columns].copy()
numeric_columns = table.select_dtypes(include='number').columns
table[numeric_columns] = table[numeric_columns].round(4)
display(table)

## Curvas de validation y comparación en test limpio

In [ ]:
display(Image(filename=str(result_dir / 'validation_far_vs_frr.png'), width=700))
display(Image(filename=str(result_dir / 'validation_f1_vs_threshold.png'), width=700))
display(Image(filename=str(result_dir / 'test_candidate_comparison.png'), width=850))

## Threshold recomendado y stress separado

La demo RFID + rostro usa `security_first`. Las métricas de stress no participaron en la selección.

In [ ]:
recommended = comparison.loc[comparison['criterion'] == 'security_first'].iloc[0]
print(f"Threshold demo: {recommended['threshold']:.10f}")
display(comparison.loc[comparison['criterion'] == 'security_first', columns])
display(stress.loc[stress['criterion'] == 'security_first', ['condition', 'accuracy', 'f1', 'far', 'frr', 'fp', 'fn']])

## Interpretación

- `max_f1` mantiene la referencia histórica, pero no controla directamente el FAR.
- `security_first` (`0.3128704727`) es la recomendación operativa para la demo.
- `balanced_far_frr` (`0.1535217762`) es una alternativa menos estricta.
- `far_lte_1` (`0.8299853802`) demuestra que el dataset actual no permite FAR ≤ 1% sin un FRR excesivo; no se recomienda para operación.
- Poca luz continúa siendo la condición más débil y requiere datos reales y control de calidad de captura.